# 01 · Control de calidad y preprocesamiento
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kur1sutaru/ISMB-LATAM/blob/main/notebooks/es/01_quality_control_preprocessing.ipynb)

**De Células a Clústeres: Introducción al Análisis de Single-Cell RNA-seq en Python**  
ISCB-LATAM SolBio SPBBC 2026 · Lima, Perú · 9 de noviembre de 2026  
Instructora: Dra. Cristal Villalba Silva (UT Health San Antonio) · [github.com/Kur1sutaru](https://github.com/Kur1sutaru)

⏱ **Tiempo en el tutorial:** 9:40–10:15 (35 min)

**Al final de este notebook podrás:**
- Calcular y graficar métricas de calidad por célula (UMIs, genes, % mitocondrial, % ribosomal)
- Elegir umbrales de filtrado: cortes fijos vs. cortes basados en los datos (MAD)
- Detectar y eliminar dobletes con Scrublet
- Normalizar, transformar a log y seleccionar genes altamente variables

> 🟢 Las secciones **Core** se ven en vivo. 🔎 Las secciones **Extra** son para estudiar después del tutorial.

## 0. Configuración 🟢
Ejecuta primero la celda de abajo (▶ o `Shift + Enter`). En Google Colab instala los paquetes (~1 min);
en tu computadora asume que ya ejecutaste `pip install -r requirements.txt`.

**Opcional:** cambia `USE_DRIVE = True` para guardar tus resultados en Google Drive y reutilizarlos en
los siguientes notebooks. Si lo dejas en `False`, cada notebook reconstruye lo que necesita automáticamente.

In [ ]:
import os, sys, urllib.request
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q "scanpy[leiden,scrublet]>=1.10"

USE_DRIVE = False  # True = guardar resultados en Google Drive
WORKDIR = "sc_tutorial"
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/sc_tutorial"
os.makedirs(WORKDIR, exist_ok=True)

# Funciones auxiliares del tutorial (descarga + checkpoints)
if os.path.exists("../../scripts/sc_tutorial.py"):          # ejecutando desde un clon local
    sys.path.insert(0, os.path.abspath("../../scripts"))
elif not os.path.exists("sc_tutorial.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Kur1sutaru/ISMB-LATAM/main/scripts/sc_tutorial.py", "sc_tutorial.py")
import sc_tutorial as tut

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)
sc.settings.figdir = os.path.join(WORKDIR, "figures")
print("scanpy", sc.__version__, "| working directory:", os.path.abspath(WORKDIR))

## 1. Cargar la matriz de Cell Ranger 🟢
`tut.load_pbmc3k_raw()` hace lo que hicimos paso a paso en el notebook 00 (descarga + `sc.read_10x_mtx`).

In [ ]:
adata = tut.load_pbmc3k_raw(WORKDIR)
adata

## 2. ¿Por qué control de calidad? 🟢
No todos los códigos de barras son una célula individual sana:

| Problema | Señal | Por qué ocurre |
|---|---|---|
| **Célula muriendo / rota** | pocos genes, alto **% de lecturas mitocondriales** | el mRNA citoplasmático se escapa, los transcritos mitocondriales quedan |
| **Gota vacía** con RNA ambiental | muy pocos UMIs y genes | Cell Ranger la conservó como célula |
| **Doblete** (dos células en una gota) | demasiados UMIs/genes, marcadores mezclados | ~0,8 % por cada 1.000 células cargadas |
| Contaminación con glóbulos rojos | alta **hemoglobina** (`HBA1`, `HBB`) | común en sangre y tejidos |

Las detectamos con algunos números por célula.

In [ ]:
# Marcar grupos de genes (símbolos humanos; en ratón usa "mt-", ("Rps", "Rpl"), "^Hb[^(p)]")
adata.var["mt"] = adata.var_names.str.startswith("MT-")
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
adata.var["hb"] = adata.var_names.str.contains(r"^HB[^(P)]")
print(adata.var[["mt", "ribo", "hb"]].sum())

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)
adata.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt", "pct_counts_ribo", "pct_counts_hb"]].describe().round(2)

### Mirar las distribuciones 🟢

In [ ]:
sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo"],
             jitter=0.4, multi_panel=True)

In [ ]:
sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts", color="pct_counts_mt")

**Cómo leerlo:** las células sanas están en la diagonal (más UMIs → más genes). Las células **abajo a la izquierda**
con **alto % MT** (amarillo) son de baja calidad. Las que están muy **arriba a la derecha** pueden ser dobletes.

## 3. Elegir umbrales 🟢
**Opción A: cortes fijos.** Simple y común para muestras conocidas. Para PBMC 3k la elección clásica (tutoriales
de Seurat y Scanpy) es: ≥ 200 genes, < 2.500 genes, < 5 % mitocondrial.

**Opción B: cortes basados en los datos (MAD).** Marca las células a más de *k* **desviaciones absolutas medianas**
de la mediana. Se adapta a cada muestra, útil cuando tienes muchas muestras o tejidos con valores basales
distintos (p. ej. neuronas o cardiomiocitos tienen naturalmente más % MT).

In [ ]:
def is_outlier(adata, metric, nmads):
    x = adata.obs[metric]
    mad = np.median(np.abs(x - np.median(x)))
    return (x < np.median(x) - nmads * mad) | (x > np.median(x) + nmads * mad)

adata.obs["outlier_mad"] = (
    is_outlier(adata, "log1p_total_counts", 5)
    | is_outlier(adata, "log1p_n_genes_by_counts", 5)
    | is_outlier(adata, "pct_counts_mt", 3)
)
adata.obs["outlier_fixed"] = (
    (adata.obs.n_genes_by_counts < 200) | (adata.obs.n_genes_by_counts >= 2500) | (adata.obs.pct_counts_mt >= 5)
)
pd.crosstab(adata.obs["outlier_fixed"], adata.obs["outlier_mad"],
            rownames=["fixed cut-offs"], colnames=["MAD cut-offs"])

Ambas estrategias coinciden casi siempre. Seguimos con los cortes **fijos** para que tus números coincidan con
los tutoriales clásicos. ¡Siempre reporta los umbrales que usaste en la sección de métodos!

In [ ]:
n0 = adata.n_obs
sc.pp.filter_cells(adata, min_genes=200)   # células con < 200 genes
sc.pp.filter_genes(adata, min_cells=3)     # genes vistos en < 3 células
adata = adata[(adata.obs.n_genes_by_counts < 2500) & (adata.obs.pct_counts_mt < 5)].copy()
print(f"cells: {n0} -> {adata.n_obs}   genes: {adata.n_vars}")

In [ ]:
sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"], jitter=0.4, multi_panel=True)

## 4. Dobletes con Scrublet 🟢
Scrublet simula dobletes artificiales sumando pares aleatorios de células y luego puntúa cada célula real según
cuánto se parece a los dobletes simulados. Funciona **por muestra** (usa `batch_key="sample"` si tienes varias).

In [ ]:
sc.pp.scrublet(adata, random_state=0)
print(adata.obs["predicted_doublet"].value_counts())
sc.pl.scrublet_score_distribution(adata)

In [ ]:
adata = adata[~adata.obs["predicted_doublet"]].copy()
print("cells after doublet removal:", adata.n_obs)

> 🔎 **Extra: RNA ambiental.** El mRNA libre de células lisadas termina en todas las gotas. Herramientas como
> **SoupX** (R), **CellBender** (Python, GPU) o **scAR** estiman esa "sopa" a partir de `raw_feature_bc_matrix`
> y la restan. Importa sobre todo en tejidos con transcritos muy abundantes (p. ej. epitelios, hemoglobina en sangre).

## 5. Normalización 🟢
Las células se secuenciaron a distintas profundidades, así que los conteos crudos no son comparables. El enfoque estándar:
1. **Guardar los conteos crudos** en una capa (los necesitamos luego para la expresión diferencial pseudobulk).
2. **Escalar** cada célula al mismo total (10.000 conteos = "conteos por 10k").
3. **Transformar a log**, `log(x + 1)`, para reducir el efecto de los genes muy expresados.

Alternativas: agrupamiento de scran, residuos de Pearson analíticos (`sc.experimental.pp.normalize_pearson_residuals`)
o `SCTransform` en Seurat.

In [ ]:
adata.layers["counts"] = adata.X.copy()   # conteos crudos, nunca se modifican
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata  # copia de los valores log-normalizados de TODOS los genes

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].hist(np.asarray(adata.layers["counts"].sum(1)).ravel(), bins=50, color="#264653")
axes[0].set(title="raw counts per cell", xlabel="total")
axes[1].hist(np.asarray(adata.X.sum(1)).ravel(), bins=50, color="#2a9d8f")
axes[1].set(title="after normalize + log1p", xlabel="total")
plt.tight_layout()

## 6. Genes altamente variables (HVGs) 🟢
La mayoría de los genes no se expresan o son constantes entre células: agregan ruido pero no información.
Nos quedamos con los ~2.000 genes cuya varianza es mayor de lo esperado para su expresión media.
Solo se **marcan** (`adata.var["highly_variable"]`); todos los genes siguen en el objeto.

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
sc.pl.highly_variable_genes(adata)
print(adata.var.query("highly_variable").sort_values("dispersions_norm", ascending=False).head(15).index.tolist())

## 7. Guardar el checkpoint 🟢

In [ ]:
os.makedirs(f"{WORKDIR}/results", exist_ok=True)
adata.write_h5ad(f"{WORKDIR}/results/pbmc3k_qc.h5ad")
adata

### 💡 Te toca a ti
Vuelve a cargar los datos crudos (`tut.load_pbmc3k_raw(WORKDIR)`) y cuenta cuántas células pasarían si permitieras
hasta **10 %** de lecturas mitocondriales en vez de 5 %. ¿Es una gran diferencia para PBMCs?

In [ ]:
# Tu código aquí

<details><summary><b>Ver solución</b></summary>

```python
raw = tut.load_pbmc3k_raw(WORKDIR)
raw.var["mt"] = raw.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(raw, qc_vars=["mt"], inplace=True)
for thr in (5, 10):
    keep = (raw.obs.n_genes_by_counts >= 200) & (raw.obs.n_genes_by_counts < 2500) & (raw.obs.pct_counts_mt < thr)
    print(thr, "% MT ->", keep.sum(), "cells")
```
</details>

---
## ✅ ¡Listo!
Tus datos están limpios y normalizados. Siguiente: **02 · Reducción de dimensionalidad y clustering**.